# Tutorial 05 — Preference optimization and reinforcement learning for reasoning

[Lecture notes](../lectures/05_preferences.md) · [Exercise sheet](../exercises/05_preferences.md)

**40 minutes, CPU, no accounts or model downloads.** Requires Python 3.10+ and NumPy. These are mechanism experiments using authored or synthetic data, not frontier benchmark results. Restart the kernel and run all cells in order.

**Objective:** Optimize a two-response policy with DPO; calculate group-relative advantages and expose a zero-variance reward group.

Allocate 5 minutes to setup, 15 to the mechanism, 10 to interventions, and 10 to discussion. Record predictions before running. Complete the separate exercise sheet after class.


In [1]:
import sys
import numpy as np
rng = np.random.default_rng(42)
print("Python", sys.version.split()[0], "NumPy", np.__version__, "seed", 42)


Python 3.13.1 NumPy 2.1.3 seed 42


## 1. DPO over a two-response policy

One scalar controls the probability of preferred response A over B. The reference is fixed. This tiny policy makes the objective directly inspectable.

Before running: predict the output or the invariant being checked.


In [2]:
def sigmoid(x): return 1/(1+np.exp(-x))
def dpo_loss(delta,reference,beta): return np.logaddexp(0,-beta*(delta-reference))
reference=0.0; theta=0.0; beta=2.0
assert np.isclose(dpo_loss(theta,reference,beta),np.log(2))
for step in range(51):
    z=beta*(theta-reference)
    grad=beta*(sigmoid(z)-1)
    theta-=.1*grad
    if step%10==0: print(step,"loss",dpo_loss(theta,reference,beta),"P(A)",sigmoid(theta))
assert sigmoid(theta)>.5


0 loss 0.5981388693815918 P(A) 0.52497918747894
10 loss 0.21510410081871448 P(A) 0.6711910596963976
20 loss 0.12266300827760743 P(A) 0.7346175540499525
30 loss 0.08449488967546502 P(A) 0.7710517790558927
40 loss 0.0640873839281572 P(A) 0.7953769206614584
50 loss 0.05148720776170229 P(A) 0.8131005364991026


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## 2. Conflicting preferences

A 70/30 annotation split has an interior optimum for the preference probability, unlike all-consistent preferences. Do not confuse preference probability sigmoid(beta*theta) with response probability sigmoid(theta).

Before running: predict the output or the invariant being checked.


In [3]:
labels=np.array([1]*7+[0]*3); theta_noisy=0.0
for _ in range(200):
    pred=sigmoid(beta*(theta_noisy-reference))
    theta_noisy-=.1*beta*(pred-labels).mean()
print("preference probability",sigmoid(beta*theta_noisy),"response probability",sigmoid(theta_noisy))
assert np.isclose(sigmoid(beta*theta_noisy),.7,atol=.01)


preference probability 0.6999999963392637 response probability 0.6043560741769496


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## 3. Group-relative advantages

This computes one ingredient used by group-relative policy optimization, not a complete RL algorithm. The second group has no relative learning signal.

Before running: predict the output or the invariant being checked.


In [4]:
rewards=np.array([[0.,1.,1.,0.],[1.,1.,1.,1.]])
advantages=(rewards-rewards.mean(axis=1,keepdims=True))/(rewards.std(axis=1,keepdims=True)+1e-8)
print(advantages)
assert np.allclose(advantages.mean(axis=1),0)
assert np.allclose(advantages[1],0)
print("worked-example loss",dpo_loss(.4,.1,2))


[[-0.99999998  0.99999998  0.99999998 -0.99999998]
 [ 0.          0.          0.          0.        ]]
worked-example loss 0.4374879504858856


**Interpretation checkpoint:** Describe the observed result, identify which line implements the key assumption, and propose one change that could invalidate your conclusion. Keep the printed outputs as evidence.


## Practical discussion

1. Can DPO improve preference loss while worsening factuality?
2. Why is a group of identical rewards uninformative for standardized group advantages?
3. What happens if the chosen/rejected labels are reversed?


## Extension and submission

Collect paired outputs from `extensions/hf_text.py` under a fixed human rubric; do not use the same examples to tune and report preference agreement.

See the [extension guide](../extensions/README.md) before running model downloads. Extensions require additional dependencies and are not necessary for full exercise credit.

Submit the completed notebook with a 150–250-word interpretation and the separate exercise answers. Include a baseline, one controlled change, data provenance, seed, and an honest limitation. Never replace measured results with expected trends.
